# 04 · Modelos · Akira — Redes neuronales (MLP)

Clasificación binaria sobre el dataset **Bank Marketing**: predecir si un cliente contratará un depósito a plazo (`y = 1`) o no (`y = 0`).

En este notebook se entrenan **4 perceptrones multicapa** (`MLPClassifier` de scikit-learn) —un modelo base y 3 experimentos— y cada uno se evalúa **con y sin SMOTE**, lo que da **8 ejecuciones** registradas en MLflow.

## 0. Librerías

In [2]:
import sys
import warnings
from pathlib import Path

import mlflow
import pandas as pd
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline  # Pipeline de imblearn, NO el de sklearn
from sklearn.compose import ColumnTransformer
from sklearn.exceptions import ConvergenceWarning
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Raíz del proyecto (el notebook se ejecuta desde notebooks/)
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(RAIZ))
from src.utils import EXPERIMENTO, MLFLOW_URI  # mismo experimento MLflow que el resto del equipo

SEMILLA = 42
warnings.filterwarnings("ignore", category=ConvergenceWarning)

## 1. Carga de los datos

Se cargan directamente los datos de entrenamiento ya procesados en `01_eda.ipynb` (`data/processed`), los mismos que usa el resto del equipo. En ese notebook ya se eliminó `duration` (fuga de datos), se pasó `y` a 1/0, se trataron los nulos, se crearon las variables nuevas y se hizo la división estratificada 80/20.

Aquí solo se usa el conjunto de **train**: el test no se toca durante la validación cruzada.

In [3]:
X_train = pd.read_csv("../data/processed/X_train.csv")
y_train = pd.read_csv("../data/processed/y_train.csv").values.ravel()

print("X_train:", X_train.shape, "| y_train:", y_train.shape)
print("Proporción de positivos en train: {:.3f}".format(y_train.mean()))

X_train: (32940, 21) | y_train: (32940,)
Proporción de positivos en train: 0.113


## 2. Preprocesado

Un `ColumnTransformer` que aplica:

- **`StandardScaler`** a las variables numéricas. Las redes neuronales se entrenan por descenso de gradiente y son muy sensibles a la escala: sin escalar, las variables con rangos grandes (p. ej. `nr.employed` ≈ 5000) dominarían a las pequeñas y la convergencia sería lenta e inestable.
- **`OneHotEncoder(handle_unknown='ignore')`** a las categóricas, para que una categoría no vista en entrenamiento no provoque un error (se codifica como todo ceros).

El preprocesador **no se ajusta aquí**: se mete dentro del pipeline para que en cada pliegue de la validación cruzada se ajuste solo con los datos de entrenamiento de ese pliegue.

In [4]:
cols_numericas = X_train.select_dtypes(include="number").columns.tolist()
cols_categoricas = X_train.select_dtypes(exclude="number").columns.tolist()

print("Numéricas:", cols_numericas)
print("Categóricas:", cols_categoricas)


def crear_preprocesador() -> ColumnTransformer:
    """Devuelve un ColumnTransformer nuevo (sin ajustar) para cada pipeline."""
    return ColumnTransformer([
        ("num", StandardScaler(), cols_numericas),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cols_categoricas),
    ])

Numéricas: ['age', 'campaign', 'pdays', 'previous', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed', 'contactado_antes', 'contactos_totales']
Categóricas: ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'poutcome']


## 3. Modelos MLP: justificación de los experimentos

**Modelo base — `mlp_base`**: una sola capa oculta de 100 neuronas, activación `relu`, `alpha = 0.0001` (los valores por defecto de scikit-learn). Sirve como punto de referencia contra el que comparar los cambios. En todos los modelos se usa `early_stopping=True` (se reserva un 10 % del pliegue de entrenamiento como validación interna y se detiene el entrenamiento cuando deja de mejorar), lo que evita entrenar épocas de más y reduce el sobreajuste y el tiempo de cómputo.

**Experimento 1 — Más capas ocultas (`mlp_profundo`, `hidden_layer_sizes=(128, 64, 32)`)**
Una red más profunda puede aprender **representaciones jerárquicas** y relaciones no lineales más complejas entre variables (p. ej. la interacción entre indicadores macroeconómicos como `euribor3m` o `emp.var.rate` y el perfil del cliente). La estructura en embudo (128 → 64 → 32) va comprimiendo la información hacia lo más relevante. Con este experimento comprobamos si **aumentar la capacidad** del modelo mejora la detección de clientes que contratan, o si por el contrario provoca sobreajuste y solo aumenta el tiempo de entrenamiento.

**Experimento 2 — Activación `tanh` (`mlp_tanh`)**
`relu` es rápida, pero las neuronas pueden "morir" (salida siempre 0 para entradas negativas, gradiente nulo). `tanh` es **suave, acotada en (−1, 1) y centrada en cero**, lo que encaja bien con datos estandarizados (media 0) y produce gradientes más estables en redes poco profundas. Además, con muchas variables binarias del one-hot, la salida centrada en cero puede facilitar la optimización. Comparamos si cambiar la no linealidad, manteniendo la arquitectura del modelo base, modifica el rendimiento.

**Experimento 3 — Mayor regularización (`mlp_regularizado`, `alpha=0.01`)**
`alpha` es el coeficiente de la **penalización L2** sobre los pesos. Aumentarlo 100 veces obliga a la red a mantener pesos pequeños y produce fronteras de decisión más suaves, **reduciendo la varianza y el sobreajuste**. Es especialmente relevante aquí porque la clase positiva es minoritaria (≈ 11 %) y, sobre todo al usar SMOTE (que genera ejemplos sintéticos), la red podría memorizar patrones artificiales. Con este experimento evaluamos si una red más regularizada **generaliza mejor** en validación cruzada.

Cada uno de los 3 experimentos cambia **un único aspecto** respecto al modelo base, para poder atribuir a ese cambio cualquier diferencia en las métricas.

In [5]:
params_comunes = dict(
    max_iter=300,
    early_stopping=True,
    random_state=SEMILLA,
)

modelos = {
    "mlp_base": MLPClassifier(
        hidden_layer_sizes=(100,), activation="relu", alpha=0.0001, **params_comunes
    ),
    "mlp_profundo": MLPClassifier(
        hidden_layer_sizes=(128, 64, 32), activation="relu", alpha=0.0001, **params_comunes
    ),
    "mlp_tanh": MLPClassifier(
        hidden_layer_sizes=(100,), activation="tanh", alpha=0.0001, **params_comunes
    ),
    "mlp_regularizado": MLPClassifier(
        hidden_layer_sizes=(100,), activation="relu", alpha=0.01, **params_comunes
    ),
}

for nombre, modelo in modelos.items():
    print(f"{nombre:<18} capas={modelo.hidden_layer_sizes}, activación={modelo.activation}, alpha={modelo.alpha}")

mlp_base           capas=(100,), activación=relu, alpha=0.0001
mlp_profundo       capas=(128, 64, 32), activación=relu, alpha=0.0001
mlp_tanh           capas=(100,), activación=tanh, alpha=0.0001
mlp_regularizado   capas=(100,), activación=relu, alpha=0.01


## 4. Pipeline, SMOTE, validación cruzada y registro en MLflow

Cada uno de los 4 modelos se evalúa **con y sin SMOTE** → **8 ejecuciones**.

**¿Por qué el `Pipeline` de `imblearn` y no el de `sklearn`?** El de sklearn no admite pasos que *remuestrean* (cambian el número de filas), como SMOTE. El de imblearn sí, y además aplica SMOTE **solo durante el `fit`**: en cada pliegue de `cross_validate` se generan ejemplos sintéticos únicamente a partir de los datos de entrenamiento de ese pliegue, mientras que el pliegue de validación se evalúa con sus datos reales, sin tocar. Si se aplicara SMOTE antes de la validación cruzada, los ejemplos sintéticos (interpolados a partir de vecinos) se mezclarían con los pliegues de validación y las métricas saldrían infladas: **fuga de datos**.

El orden de los pasos es: `preprocesado` → `smote` (si toca) → `modelo`. SMOTE va después del preprocesado porque trabaja con distancias entre vecinos y necesita variables numéricas y escaladas.

**Evaluación**: `cross_validate` con 5 pliegues estratificados sobre **train** (el test no se toca), extrayendo `recall`, `precision`, `f1` y `fit_time`. Interesa sobre todo el **recall** de la clase positiva (no perder clientes que contratarían) sin hundir la precisión, por eso el F1 es la métrica de equilibrio.

**MLflow**: cada ejecución guarda los hiperparámetros del MLP, si se usó SMOTE, y la media (y desviación) de las métricas de validación.

In [6]:
mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment(EXPERIMENTO)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA)
scoring = {"recall": "recall", "precision": "precision", "f1": "f1"}
params_a_registrar = ["hidden_layer_sizes", "activation", "alpha", "solver",
                      "learning_rate_init", "max_iter", "early_stopping", "random_state"]

resultados = []

for nombre, modelo in modelos.items():
    for usar_smote in [False, True]:
        # Construimos el pipeline: preprocesado -> (SMOTE) -> modelo
        pasos = [("preprocesado", crear_preprocesador())]
        if usar_smote:
            pasos.append(("smote", SMOTE(random_state=SEMILLA)))
        pasos.append(("modelo", modelo))
        pipeline = Pipeline(pasos)

        # Validación cruzada (cv=5) solo sobre el set de entrenamiento
        cv_res = cross_validate(pipeline, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)

        metricas = {
            "recall": cv_res["test_recall"].mean(),
            "precision": cv_res["test_precision"].mean(),
            "f1": cv_res["test_f1"].mean(),
            "fit_time": cv_res["fit_time"].mean(),
            "recall_std": cv_res["test_recall"].std(),
            "precision_std": cv_res["test_precision"].std(),
            "f1_std": cv_res["test_f1"].std(),
        }

        nombre_run = f"{nombre}_{'smote' if usar_smote else 'sin_smote'}"

        # Registro en MLflow
        with mlflow.start_run(run_name=nombre_run):
            params = {p: modelo.get_params()[p] for p in params_a_registrar}
            params["modelo"] = nombre
            params["smote"] = usar_smote
            params["cv_folds"] = 5
            mlflow.log_params(params)
            mlflow.log_metrics({k: float(v) for k, v in metricas.items()})
            mlflow.set_tag("autor", "akira")
            mlflow.set_tag("familia", "MLP")

        resultados.append({"modelo": nombre, "smote": usar_smote, **metricas})
        print(f"{nombre_run:<28} recall={metricas['recall']:.3f}  precision={metricas['precision']:.3f}  "
              f"f1={metricas['f1']:.3f}  fit_time={metricas['fit_time']:.1f}s")

2026/11/10 16:26:13 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/11/10 16:26:13 INFO mlflow.store.db.utils: Updating database tables
2026/11/10 16:26:15 INFO mlflow.tracking.fluent: Experiment with name 'reto1-deposito-banco' does not exist. Creating a new experiment.


mlp_base_sin_smote           recall=0.244  precision=0.653  f1=0.354  fit_time=2.0s
mlp_base_smote               recall=0.405  precision=0.337  f1=0.367  fit_time=15.5s
mlp_profundo_sin_smote       recall=0.225  precision=0.672  f1=0.336  fit_time=3.3s
mlp_profundo_smote           recall=0.347  precision=0.340  f1=0.342  fit_time=18.3s
mlp_tanh_sin_smote           recall=0.269  precision=0.632  f1=0.377  fit_time=2.2s
mlp_tanh_smote               recall=0.397  precision=0.319  f1=0.353  fit_time=17.8s
mlp_regularizado_sin_smote   recall=0.248  precision=0.656  f1=0.359  fit_time=1.6s
mlp_regularizado_smote       recall=0.444  precision=0.347  f1=0.388  fit_time=10.7s


## 5. Comparativa de resultados

Tabla resumen de las 8 ejecuciones (media de los 5 pliegues de validación), ordenada por F1.

In [7]:
df_resultados = (
    pd.DataFrame(resultados)[["modelo", "smote", "recall", "precision", "f1", "fit_time"]]
    .sort_values("f1", ascending=False)
    .reset_index(drop=True)
)
df_resultados.style.format({"recall": "{:.3f}", "precision": "{:.3f}", "f1": "{:.3f}", "fit_time": "{:.1f}"})

,modelo,smote,recall,precision,f1,fit_time
0,mlp_regularizado,True,0.444,0.347,0.388,10.7
1,mlp_tanh,False,0.269,0.632,0.377,2.2
2,mlp_base,True,0.405,0.337,0.367,15.5
3,mlp_regularizado,False,0.248,0.656,0.359,1.6
4,mlp_base,False,0.244,0.653,0.354,2.0
5,mlp_tanh,True,0.397,0.319,0.353,17.8
6,mlp_profundo,True,0.347,0.340,0.342,18.3
7,mlp_profundo,False,0.225,0.672,0.336,3.3


Las 8 ejecuciones quedan registradas en el experimento de MLflow del proyecto. Para verlas, desde la raíz del proyecto, ejecuta: **`mlflow ui --backend-store-uri sqlite:///mlflow.db`**

Lo esperable es que **SMOTE aumente claramente el recall** (el modelo ve muchos más ejemplos positivos y se atreve a predecir la clase minoritaria) a costa de **bajar la precisión**, y que el F1 indique cuál de las dos opciones ofrece mejor equilibrio para cada arquitectura.